# PathVQA-TR — Kaggle Training
ViT-B/16 + BERT + MLP Fusion, freeze encoders, yes/no classification

In [ ]:
!pip install -q datasets transformers torch torchvision tqdm

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import ViTModel, BertModel, ViTImageProcessor, BertTokenizer
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
import io

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
ANSWER_TO_IDX = {'yes': 0, 'no': 1}

class PathVQADataset(Dataset):
    def __init__(self, split, image_processor, tokenizer, max_length=128):
        raw = load_dataset('flaviagiammarino/path-vqa', split=split)
        self.data = [item for item in raw if str(item['answer']).strip().lower() in ANSWER_TO_IDX]
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item['image']
        if isinstance(image, bytes):
            image = Image.open(io.BytesIO(image)).convert('RGB')
        elif not isinstance(image, Image.Image):
            image = Image.fromarray(image).convert('RGB')
        else:
            image = image.convert('RGB')
        pixel_values = self.image_processor(images=image, return_tensors='pt')['pixel_values'].squeeze(0)
        enc = self.tokenizer(str(item['question']), max_length=self.max_length,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {
            'pixel_values': pixel_values,
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(ANSWER_TO_IDX[str(item['answer']).strip().lower()], dtype=torch.long)
        }

In [ ]:
class PathVQAModel(nn.Module):
    def __init__(self, num_classes=2, hidden_dim=512, dropout=0.3):
        super().__init__()
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224')
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        for p in self.vit.parameters(): p.requires_grad = False
        for p in self.bert.parameters(): p.requires_grad = False
        self.fusion = nn.Sequential(
            nn.Linear(768 + 768, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        img = self.vit(pixel_values=pixel_values).pooler_output
        txt = self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        return self.fusion(torch.cat([img, txt], dim=-1))

In [ ]:
BATCH_SIZE = 32
EPOCHS = 5
LR = 3e-4

image_processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_ds = PathVQADataset('train', image_processor, tokenizer)
val_ds = PathVQADataset('validation', image_processor, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
model = PathVQAModel().to(DEVICE)
optimizer = torch.optim.AdamW(model.fusion.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}'):
        pv = batch['pixel_values'].to(DEVICE)
        ii = batch['input_ids'].to(DEVICE)
        am = batch['attention_mask'].to(DEVICE)
        lb = batch['label'].to(DEVICE)
        optimizer.zero_grad()
        logits = model(pv, ii, am)
        loss = criterion(logits, lb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (logits.argmax(1) == lb).sum().item()
        total += lb.size(0)
    
    model.eval()
    vc, vt, vl = 0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            pv = batch['pixel_values'].to(DEVICE)
            ii = batch['input_ids'].to(DEVICE)
            am = batch['attention_mask'].to(DEVICE)
            lb = batch['label'].to(DEVICE)
            logits = model(pv, ii, am)
            vl += criterion(logits, lb).item()
            vc += (logits.argmax(1) == lb).sum().item()
            vt += lb.size(0)
    
    val_acc = vc / vt
    scheduler.step()
    print(f'Epoch {epoch} | train_acc={correct/total:.4f} | val_acc={val_acc:.4f}')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  -> Saved (val_acc={best_acc:.4f})')

print('Done. Best val_acc:', best_acc)